# Solution E1 · The early stopping that never stops

**Type**: Find the bug

## Objective

Understand why a colleague, who turned on `early_stopping_rounds=30` and "monitors both curves" by passing training and validation to `eval_set`, always gets `best_iteration = 499` (the model uses all the rounds) and concludes that "early stopping is not needed, the model always improves". Find **which set XGBoost uses
to stop**, fix it and measure the difference on the test set.

In [ ]:
import sys, pathlib
_root = next(p for p in [pathlib.Path.cwd(), *pathlib.Path.cwd().parents] if (p / "common").is_dir())
sys.path.insert(0, str(_root))

import warnings; warnings.filterwarnings("ignore")     # clean output: here the warnings are just noise
import numpy as np, pandas as pd
import matplotlib.pyplot as plt
from common import datasets as ds
from common.verify import check, close, done, verify

import xgboost as xgb
from sklearn.datasets import make_classification
from sklearn.metrics import log_loss
from sklearn.model_selection import train_test_split

X, y = make_classification(5000, n_features=12, n_informative=5, class_sep=0.7, flip_y=0.1, random_state=0)
X_tr, X_rest, y_tr, y_rest = train_test_split(X, y, test_size=0.5, random_state=0)
X_val, X_te, y_val, y_te = train_test_split(X_rest, y_rest, test_size=0.5, random_state=0)


def new_model():
    return xgb.XGBClassifier(n_estimators=500, learning_rate=0.1, max_depth=5, eval_metric="logloss", early_stopping_rounds=30, n_jobs=-1, random_state=0)

## Commented solution

In [ ]:
m_bug = new_model()
m_bug.fit(X_tr, y_tr, eval_set=[(X_val, y_val), (X_tr, y_tr)], verbose=False)      # 'I monitor validation and training'
best_col = m_bug.best_iteration + 1
bug_test_ll = log_loss(y_te, m_bug.predict_proba(X_te)[:, 1])
print(f"rounds used: {best_col} out of 500 | test log-loss {bug_test_ll:.4f} -> 'early stopping is not needed, the model always improves'")

**Where the bug is.** With several sets in `eval_set`, XGBoost applies `early_stopping_rounds` **to the last one**. Here the last one is the training set, whose log-loss drops at every round: the criterion "no improvement in 30 rounds" never triggers and the model goes on up to 500 rounds, well past the validation minimum. No error,
no warning.

In [ ]:
which_set = "training"                                                     # eval_set[-1] is (X_tr, y_tr)
why_it_never_stops = "Early stopping looks at the last eval_set, that is the training set, whose log-loss always drops"
val = np.array(m_bug.evals_result()["validation_0"]["logloss"])              # validation_0 = first set = validation
val_min_round = int(val.argmin()) + 1
print(f"in the colleague's run the validation has its minimum at round {val_min_round}, then rises up to {val[-1]:.3f} (minimum {val.min():.3f})")

In [ ]:
m_ok = new_model()
m_ok.fit(X_tr, y_tr, eval_set=[(X_tr, y_tr), (X_val, y_val)], verbose=False)     # the validation set is the last one
ok_round = m_ok.best_iteration + 1
ok_test_ll = log_loss(y_te, m_ok.predict_proba(X_te)[:, 1])
val_ok = np.array(m_ok.evals_result()[list(m_ok.evals_result())[-1]]["logloss"])
matches = bool(int(val_ok.argmin()) == m_ok.best_iteration)
gain = bug_test_ll - ok_test_ll
print(f"rounds used: {ok_round} | test log-loss {ok_test_ll:.4f} (before {bug_test_ll:.4f}, gain {gain:.4f}) | argmin of the validation = best_iteration: {matches}")

## Check

In [ ]:
with verify():
    check(best_col >= 490, "the colleague sees that the model uses almost all the 500 rounds", "don't modify the colleague's code")
    check(which_set == "training" and isinstance(why_it_never_stops, str) and len(why_it_never_stops) > 10, "XGBoost uses the last set of eval_set, that is the training set, and you explain why", "XGBoost stops on eval_set[-1]")
    check(val_min_round is not None and 20 <= val_min_round <= 80, "the validation, in the colleague's run, had its minimum between 20 and 80 rounds: from there on the model was getting worse", "argmin of evals_result()['validation_0']['logloss'] + 1")
    check(ok_round is not None and ok_round < 100 and ok_test_ll < 0.42, "with the validation set last the model stops under 100 rounds and the test log-loss drops below 0.42", "eval_set=[(X_tr, y_tr), (X_val, y_val)]")
    check(matches is True and gain is not None and gain > 0.05, "the minimum of the validation curve matches best_iteration; the gain on the test log-loss exceeds 0.05", "bug_test_ll - ok_test_ll")
done()

## What you learned

- **Early stopping applies to the last set of `eval_set`:** with `[(val), (train)]` XGBoost looks at the training set, which always drops, and never stops (500 rounds out of 500). With the validation set last it stops at 38 rounds. There is no error or warning: the only symptom is `best_iteration` equal to `n_estimators - 1`.
- **The cost is real:** with 500 rounds the test log-loss is 0.463, with the correct early stopping 0.374: a difference of 0.09, larger than almost any improvement you would get with tuning. The validation in the wrong run had its minimum (0.396) at round 38 and then rose to 0.486 without anyone looking at it.
- **A signal to always check:** `best_iteration` very close to `n_estimators` (with learning rate 0.1) is almost always a sign that early stopping is not looking at the right curve, or that you need more `n_estimators`. In both cases it is worth plotting `evals_result()`.
- **Practical rule:** put the validation set alone or last, and use `evals_result()` to check that the minimum of its curve matches `best_iteration`. If you also want to see the training curve, put it **first**.

## Common pitfall

Adding the training set to `eval_set` "to see both curves" and putting it last: the curve under control becomes the training one and early stopping stops working.

## Interview question

> *Click on a question to show its answer.*

<details><summary>How does XGBoost choose the stopping round when `eval_set` has several sets?</summary>

It uses the last set (and, if there are several metrics, the last metric): so the validation set has to go last. If I put the training set there the stop never triggers, because its metric always improves; the sign is `best_iteration` close to `n_estimators`. I check with `evals_result()` that the minimum of the validation curve matches
`best_iteration`.

</details>